# Model-Parameter Scaling — All Datasets (combined)

Runs the **PSNR-vs-CR model-size sweep** for every dataset in one notebook and
renders a single combined figure.

For each dataset: outer sweep over the **SZ3 relative-error bound** (moves CR) ×
inner sweep over **model parameter budget** (separate curves).  CR =
`orig_bytes / (SZ3_bytes + model_bytes)` (bf16 weights, 2 B/param; aux fields are
raw side-info and **not** charged to CR).

Methodology is held uniform across datasets: **fixed `lr=1e-3`**, no early-stop,
each model trained for its full per-config wall-clock budget (an ablation that
isolates the effect of model size).

Datasets: **NYX** (512³, +5 aux), **Miranda** (1024³), **Magnetic** (512³),
**S3D** (500³ double), **Hurricane** (100×500×500, +6 aux).

Run cells top-to-bottom, or run only the dataset cells you need, then run the
final plot cell.  Each dataset frees its volumes when done.

In [1]:
import random, sys, os, glob, time, gc
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.append("/home/sam/Halo_Finder/Final_design/base_script")

import importlib
import bg_stage
importlib.reload(bg_stage)

from experiment import build_bg_only_cfg
from bg_stage import run_bg_inference, train_bg_only, unwrap_bg_model
from bg_shard import pick_bg_h_under_budget

pysz_dir = "/home/sam/Data_Compression/SZ3/tools/pysz"
if pysz_dir not in sys.path:
    sys.path.append(pysz_dir)
from pysz import SZ

def set_seed(seed=42):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | GPUs: {torch.cuda.device_count()}")

Device: cuda:0 | GPUs: 1


In [2]:
halo_finder_root = Path("/home/sam/Halo_Finder")
sz_lib_path = "/home/sam/Data_Compression/SZ3/build/lib64/libSZ3c.so"
sz_engine   = SZ(sz_lib_path)
BYTES_PER_PARAM = 2          # bf16 storage for the model weights


def build_cfg(Xs_in, Xps_in, max_train_time, bg_h, steps_per_epoch,
              lr=1e-4, epochs=200, log_prefix="", patch_size=None):
    if patch_size is None:
        patch_size = Xs_in[0].shape[2]
    cfg = build_bg_only_cfg(
        X_target=Xs_in[0], Xps=Xps_in, max_train_time=max_train_time,
        epochs=epochs, steps_per_epoch=steps_per_epoch, bg_h=bg_h,
        bg_batch=1, bg_patch_size=patch_size, lr=lr,
    )
    cfg.bg_sample_mode   = "sequential"
    cfg.bg_log_prefix    = log_prefix
    cfg.bg_arch          = "spatial"
    cfg.amp              = True
    cfg.amp_dtype        = "bf16"
    cfg.bg_ddp           = False
    cfg.bg_data_parallel = False
    return cfg


def n_params_of(model):
    return int(sum(p.numel() for p in unwrap_bg_model(model).parameters()))


def psnr_from_arrays(target, recon):
    data_range = float(target.max() - target.min())
    if data_range <= 0:
        data_range = 1.0
    mse = float(np.mean((target - recon) ** 2))
    return 100.0 if mse <= 0 else 20.0 * np.log10(data_range) - 10.0 * np.log10(mse)


print("SZ engine + helpers ready")

SZ engine + helpers ready


In [3]:
# Each loader returns: (Xs, compress_fn, orig_bytes, work_shape)
#   Xs          = [target_gt_f32, aux1_f32, ...]   (model inputs; field 0 = target)
#   compress_fn = rel -> (Xps_list, sz_bytes)      Xps_list = [target_lq_f32] + aux
#   orig_bytes  = bytes counted for CR (TARGET only; double-size for S3D)

def load_nyx():
    d = halo_finder_root / "halo_finder_v1/SDRBENCH-EXASKY-NYX-512x512x512/origin"
    shape = (512, 512, 512)
    target = "baryon_density"
    aux    = ["dark_matter_density", "temperature", "velocity_x", "velocity_y", "velocity_z"]
    lf = lambda n: np.fromfile((d / f"{n}.f32").as_posix(), dtype=np.float32).reshape(shape)
    gt = lf(target)
    aux_fields = [lf(a) for a in aux]
    Xs = [gt] + aux_fields
    orig_bytes = gt.nbytes
    def compress_fn(rel):
        b, _ = sz_engine.compress(gt, 1, 0, float(rel), 0)
        x_lq = sz_engine.decompress(b, gt.shape, np.float32).astype(np.float32)
        return [np.asarray(x_lq, np.float32)] + aux_fields, int(len(b))
    return Xs, compress_fn, orig_bytes, gt.shape


def load_miranda():
    p = (halo_finder_root / "halo_finder_v1/miranda_1024x1024x1024_float32.raw").as_posix()
    shape = (1024, 1024, 1024)
    gt = np.fromfile(p, dtype=np.float32).reshape(shape)
    def compress_fn(rel):
        b, _ = sz_engine.compress(gt, 1, 0, float(rel), 0)
        x_lq = sz_engine.decompress(b, gt.shape, np.float32).astype(np.float32)
        return [np.asarray(x_lq, np.float32)], int(len(b))
    return [gt], compress_fn, gt.nbytes, gt.shape


def load_magnetic():
    p = (halo_finder_root / "halo_finder_v1/magnetic_reconnection_512x512x512_float32.raw").as_posix()
    shape = (512, 512, 512)
    gt = np.fromfile(p, dtype=np.float32).reshape(shape)
    def compress_fn(rel):
        b, _ = sz_engine.compress(gt, 1, 0, float(rel), 0)
        x_lq = sz_engine.decompress(b, gt.shape, np.float32).astype(np.float32)
        return [np.asarray(x_lq, np.float32)], int(len(b))
    return [gt], compress_fn, gt.nbytes, gt.shape


def load_s3d():
    p = (halo_finder_root / "halo_finder_v1/s3d-500_500_500-double.raw").as_posix()
    shape = (500, 500, 500)
    gt_native = np.fromfile(p, dtype=np.float64).reshape(shape)   # double for SZ3 + CR
    gt = gt_native.astype(np.float32)                            # model trains in f32
    orig_bytes = gt_native.nbytes                                # 8 B/elem -> faithful CR
    def compress_fn(rel):
        b, _ = sz_engine.compress(gt_native, 1, 0, float(rel), 0)
        x_lq = sz_engine.decompress(b, gt_native.shape, np.float64).astype(np.float32)
        return [np.asarray(x_lq, np.float32)], int(len(b))
    return [gt], compress_fn, orig_bytes, gt.shape


def load_hurricane():
    ddir = "/home/sam/Halo_Finder/halo_finder_v1/100x500x500"
    shape = (100, 500, 500)
    target = "CLOUDf48"
    aux    = ["PRECIPf48", "Pf48", "TCf48", "Uf48", "Vf48", "Wf48"]
    aux    = [a for a in aux if os.path.isfile(os.path.join(ddir, f"{a}.bin.f32"))]
    lf = lambda n: np.fromfile(os.path.join(ddir, f"{n}.bin.f32"), dtype=np.float32).reshape(shape)
    gt = lf(target)
    aux_fields = [lf(a) for a in aux]
    Xs = [gt] + aux_fields
    orig_bytes = gt.nbytes
    def compress_fn(rel):
        b, _ = sz_engine.compress(gt, 1, 0, float(rel), 0)
        x_lq = sz_engine.decompress(b, gt.shape, np.float32).astype(np.float32)
        return [np.asarray(x_lq, np.float32)] + aux_fields, int(len(b))
    return Xs, compress_fn, orig_bytes, gt.shape


SPECS = {
    "NYX":       dict(load=load_nyx,       time_per_config=60.0,
                      rel_errs=[1e-6, 3e-6, 5e-6, 7e-6, 9e-6],
                      param_targets=[3000, 6000, 9000, 15000, 25000, 30000, 60000, 90000]),
    "Miranda":   dict(load=load_miranda,   time_per_config=120.0,
                      rel_errs=[3e-3, 5e-3, 7e-3, 9e-3, 1e-2],
                      param_targets=[30000, 100000, 150000, 240000, 360000, 500000]),
    "Magnetic":  dict(load=load_magnetic,  time_per_config=60.0,
                      rel_errs=[1e-3, 2e-3, 3e-3, 4e-3, 5e-3, 6e-3],
                      param_targets=[3000, 10000, 30000, 100000]),
    "S3D":       dict(load=load_s3d,       time_per_config=60.0,
                      rel_errs=[1e-5, 3e-5, 5e-5, 7e-5, 9e-5, 1e-4],
                      param_targets=[3000, 10000, 30000, 100000]),
    "Hurricane": dict(load=load_hurricane, time_per_config=60.0,
                      rel_errs=[1e-3, 3e-3, 5e-3, 7e-3, 1e-2, 5e-2],
                      param_targets=[3000, 6000, 10000, 30000]),
}
print("Loaders + SPECS ready:", list(SPECS))

Loaders + SPECS ready: ['NYX', 'Miranda', 'Magnetic', 'S3D', 'Hurricane']


In [4]:
FIXED_LR    = 1e-3
FREQ_WARMUP = 1

all_results = {}   # name -> dict(name, results, baselines, work_shape)


def run_sweep(name, spec):
    print(f"\n{'='*64}\n[{name}] loading...\n{'='*64}")
    t_load = time.time()
    Xs, compress_fn, orig_bytes, work_shape = spec["load"]()
    n_fields = len(Xs)
    depth, patch_sz = work_shape[0], work_shape[2]
    print(f"[{name}] loaded in {time.time()-t_load:.1f}s | shape {work_shape} | "
          f"n_fields={n_fields} | target {orig_bytes/1e6:.1f} MB")

    # budget -> largest bg_h that fits (dedup; several budgets may share a bg_h)
    seen, sweep_plan = set(), []
    for tgt in spec["param_targets"]:
        h, est_p = pick_bg_h_under_budget(tgt, shape=work_shape, n_fields=n_fields,
                                          bg_arch="spatial", h_candidates=list(range(3, 100)))
        if h not in seen:
            seen.add(h); sweep_plan.append((int(h), int(est_p)))
    sweep_plan.sort()

    rel_errs = spec["rel_errs"]
    time_per = spec["time_per_config"]
    n_runs   = len(rel_errs) * len(sweep_plan)
    print(f"[{name}] {len(rel_errs)} rel x {len(sweep_plan)} sizes = {n_runs} runs "
          f"x {time_per:.0f}s (~{n_runs*time_per/60:.0f} min) | lr={FIXED_LR:.0e}")
    for h, est_p in sweep_plan:
        print(f"    bg_h={h:2d}  ~{est_p:,} params")

    # GPU warm-up (not timed)
    _wXps, _ = compress_fn(max(rel_errs))
    _wcfg = build_cfg(Xs, _wXps, max_train_time=0.5, bg_h=4, steps_per_epoch=2,
                      lr=1e-3, epochs=1, log_prefix="warmup")
    set_seed(42)
    train_bg_only(Xs=Xs, Xps=_wXps, device=device, cfg=_wcfg,
                  evaluator=lambda m, c=_wcfg: (0.0, 0.0))
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    del _wXps

    results, baselines = [], []
    sweep_t0 = time.time()
    for rel in rel_errs:
        Xps_list, sz_bytes = compress_fn(rel)
        base_psnr = psnr_from_arrays(Xs[0], Xps_list[0])
        base_cr   = orig_bytes / sz_bytes
        baselines.append(dict(rel=rel, cr=base_cr, psnr=base_psnr))
        print(f"\n# [{name}] rel={rel:.0e} | SZ3 CR={base_cr:.2f}x | base PSNR={base_psnr:.2f} dB")

        for bg_h, est_p in sweep_plan:
            cfg = build_cfg(Xs, Xps_list, max_train_time=time_per, bg_h=bg_h,
                            steps_per_epoch=depth, lr=FIXED_LR, epochs=200,
                            log_prefix=f"{name}-r{rel:.0e}-h{bg_h}", patch_size=patch_sz)
            cfg.bg_early_stop         = False
            cfg.bg_freq_warmup_epochs = FREQ_WARMUP

            def evaluator(m, c=cfg, Xp=Xps_list, g=Xs[0], r=rel):
                xh = run_bg_inference(unwrap_bg_model(m), Xs, Xp, c, r)
                return psnr_from_arrays(g, xh), 0.0

            set_seed(42)
            model, hist = train_bg_only(Xs=Xs, Xps=Xps_list, device=device,
                                        cfg=cfg, evaluator=evaluator)
            n_params  = n_params_of(model)
            psnr_vals = [v[1] if isinstance(v, tuple) else v for v in hist.get("psnr", [])]
            psnr      = max(psnr_vals) if psnr_vals else float("nan")
            cr        = orig_bytes / (sz_bytes + n_params * BYTES_PER_PARAM)
            results.append(dict(rel=rel, bg_h=bg_h, n_params=n_params, psnr=psnr, cr=cr))
            print(f"    bg_h={bg_h:2d} ({n_params:,}p): PSNR={psnr:.2f} dB | "
                  f"CR={cr:.2f}x | +{psnr-base_psnr:.2f} dB over SZ3")
            del model
            torch.cuda.empty_cache() if torch.cuda.is_available() else None
        del Xps_list

    print(f"\n[{name}] sweep done in {(time.time()-sweep_t0)/60:.1f} min ({len(results)} models)")
    del Xs
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return dict(name=name, results=results, baselines=baselines, work_shape=work_shape)


print("run_sweep ready")

run_sweep ready


In [5]:
all_results["NYX"] = run_sweep("NYX", SPECS["NYX"])


[NYX] loading...
[NYX] loaded in 0.5s | shape (512, 512, 512) | n_fields=6 | target 536.9 MB

[Model: spatial] Total Params: 859
 [Params] Main (BG) Network : 859 parameters

[Model: spatial] Total Params: 1,396
 [Params] Main (BG) Network : 1,396 parameters

[Model: spatial] Total Params: 2,059
 [Params] Main (BG) Network : 2,059 parameters

[Model: spatial] Total Params: 2,848
 [Params] Main (BG) Network : 2,848 parameters

[Model: spatial] Total Params: 3,763
 [Params] Main (BG) Network : 3,763 parameters

[Model: spatial] Total Params: 4,804
 [Params] Main (BG) Network : 4,804 parameters

[Model: spatial] Total Params: 5,971
 [Params] Main (BG) Network : 5,971 parameters

[Model: spatial] Total Params: 7,264
 [Params] Main (BG) Network : 7,264 parameters

[Model: spatial] Total Params: 8,683
 [Params] Main (BG) Network : 8,683 parameters

[Model: spatial] Total Params: 10,228
 [Params] Main (BG) Network : 10,228 parameters

[Model: spatial] Total Params: 11,899
 [Params] Main (BG)

/home/sam/Halo_Finder/Final_design/base_script/bg_stage.py:522: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp, dtype=autocast_dtype):


warmup Epoch   1 [BG] | train_wall=2.60s | Loss: 1.000909 | Freq: 3.375000 | Global: 0.00 dB | MaxErr: 0.0
warmup [timing] first_epoch_pure_train≈3.044s (excludes this epoch's end-of-epoch eval)

warmup --- Experiment [BG_only] finished ---
warmup --- Pure training time: 3.04 s ---
warmup [timing] epochs=1 | train_wall/epoch: mean=2.60s min=2.60s max=2.60s | sum=2.60s
warmup --- Best global PSNR: 0.00 dB ---

# [NYX] rel=1e-06 | SZ3 CR=81.83x | base PSNR=129.13 dB

[Model: spatial] Total Params: 2,848
 [Params] Main (BG) Network : 2,848 parameters
NYX-r1e-06-h6 [Init] Epoch   0 | Global PSNR: 129.13 dB | MaxErr: 0.0
NYX-r1e-06-h6 [plan] pure_train_budget=60.00s | epochs_cap=200 | steps/epoch=512 | patch=512 | batch=1 | sample=sequential | data_parallel=False | amp=bf16
NYX-r1e-06-h6 [early-stop] DISABLED (cfg.bg_early_stop is False/unset)
NYX-r1e-06-h6 [gpu-sampling] 6 fields resident on cuda:0 (~6.4 GB)
NYX-r1e-06-h6 Epoch   1 [BG] | train_wall=3.23s | Loss: 0.947146 | Freq: 3.070923 

In [6]:
all_results["Miranda"] = run_sweep("Miranda", SPECS["Miranda"])


[Miranda] loading...
[Miranda] loaded in 0.6s | shape (1024, 1024, 1024) | n_fields=1 | target 4295.0 MB

[Model: spatial] Total Params: 724
 [Params] Main (BG) Network : 724 parameters

[Model: spatial] Total Params: 1,216
 [Params] Main (BG) Network : 1,216 parameters

[Model: spatial] Total Params: 1,834
 [Params] Main (BG) Network : 1,834 parameters

[Model: spatial] Total Params: 2,578
 [Params] Main (BG) Network : 2,578 parameters

[Model: spatial] Total Params: 3,448
 [Params] Main (BG) Network : 3,448 parameters

[Model: spatial] Total Params: 4,444
 [Params] Main (BG) Network : 4,444 parameters

[Model: spatial] Total Params: 5,566
 [Params] Main (BG) Network : 5,566 parameters

[Model: spatial] Total Params: 6,814
 [Params] Main (BG) Network : 6,814 parameters

[Model: spatial] Total Params: 8,188
 [Params] Main (BG) Network : 8,188 parameters

[Model: spatial] Total Params: 9,688
 [Params] Main (BG) Network : 9,688 parameters

[Model: spatial] Total Params: 11,314
 [Params]

KeyboardInterrupt: 

In [ ]:
all_results["Magnetic"] = run_sweep("Magnetic", SPECS["Magnetic"])

In [ ]:
all_results["S3D"] = run_sweep("S3D", SPECS["S3D"])

In [ ]:
all_results["Hurricane"] = run_sweep("Hurricane", SPECS["Hurricane"])

In [ ]:
# ── Combined figure: one subplot per dataset · dark colors · hollow markers ───
DARK_COLORS = ["#08306b", "#a63603", "#00441b", "#67000d", "#3f007d",
               "#525252", "#543005", "#014636", "#08519c", "#7f2704"]
MARKERS     = ["o", "s", "^", "D", "v", "P", "X", "<", ">", "h"]


def fmt_p(p):
    return f"{p/1000:.0f}k" if p >= 1000 else str(p)


def plot_dataset(ax, R, fs=15):
    results, baselines = R["results"], R["baselines"]
    psizes = sorted(set(r["n_params"] for r in results))
    for i, p in enumerate(psizes):
        pts = sorted([r for r in results if r["n_params"] == p], key=lambda r: r["cr"])
        col = DARK_COLORS[i % len(DARK_COLORS)]
        ax.plot([r["cr"] for r in pts], [r["psnr"] for r in pts],
                linestyle="-", color=col, linewidth=1.8, zorder=3,
                marker=MARKERS[i % len(MARKERS)], markersize=8,
                markerfacecolor="none", markeredgecolor=col, markeredgewidth=1.7,
                label=f"{fmt_p(p)}")
    # SZ3-only baseline (dark, hollow star)
    bl = sorted(baselines, key=lambda b: b["cr"])
    ax.plot([b["cr"] for b in bl], [b["psnr"] for b in bl],
            linestyle="--", color="black", linewidth=1.6, zorder=2,
            marker="*", markersize=12, markerfacecolor="none",
            markeredgecolor="black", markeredgewidth=1.5, label="SZ3 only")
    ax.set_xlabel("Compression Ratio", fontsize=fs)
    ax.set_ylabel("PSNR (dB)", fontsize=fs)
    ax.set_title(R["name"], fontsize=fs + 2, fontweight="bold")
    ax.tick_params(axis="both", which="major", labelsize=fs - 3)
    ax.grid(True, alpha=0.35)
    ax.legend(fontsize=fs - 5, title="model size", title_fontsize=fs - 4, loc="best")


names = [n for n in ["NYX", "Miranda", "Magnetic", "S3D", "Hurricane"] if n in all_results]
ncols = 3
nrows = (len(names) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 5.5 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax, nm in zip(axes, names):
    plot_dataset(ax, all_results[nm])
for ax in axes[len(names):]:
    ax.axis("off")

fig.suptitle("Model-Parameter Scaling — PSNR vs Compression Ratio (all datasets)",
             fontsize=18, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.97])
out_pdf = "psnr_vs_cr_all_datasets.pdf"
plt.savefig(out_pdf, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_pdf}")